# FinRise Risk Management Project - Part 2: Portfolio Construction for Surplus Liquidity

**Course:** Financial Risk Analytics  
**Team:** Umer Raza, Syed Yahya Tariq  

## Overview
Following the Risk Assessment Report, FinRise Digital Finance Ltd. requires efficient management of surplus liquidity. This notebook constructs and analyzes an investment portfolio consisting of 5 PSX-listed stocks, Treasury Bills, and Bank Deposits, while keeping the Investment Policy in mind (Annexure - A)

**Portfolio Assets:**
- **Equities (55%)**: FFC, Lucky, OGDC, SAzgar, Mebl
- **Treasury Bills (30%)**: 12-month at 15% p.a.
- **Bank Deposits (15%)**: Commercial (12% p.a.) and Islamic (11.5% p.a.), with minimum 5% in Islamic.

**Objectives:**
- Construct portfolio respecting sector and allocation limits
- Calculate expected returns, risk, and Sharpe ratio
- Optimize for maximum Sharpe ratio
- Generate summary report for CRO and CFO

## A Portfolio Overview
**Total Portfolio Value:** Rs 2,000,000,000

**Initial Diversification (before optimization):**
- Stocks (55%) - PKR 110,000,000
- Treasury Bills (30%) - PKR 60,000,000  
- Bank Deposits (15%) - PKR 30,000,000


In [352]:
portfolio_Value = 2000000000  
stocks_allocation = 0.55
treasury_allocation = 0.30
bank_allocation = 0.15
stocks_value = portfolio_Value * stocks_allocation
treasury_value = portfolio_Value * treasury_allocation
bank_value = portfolio_Value * bank_allocation
print(f"Stocks Value: ${stocks_value:,.2f}")
print(f"Treasury Value: ${treasury_value:,.2f}")
print(f"Bank Value: ${bank_value:,.2f}")


Stocks Value: $1,100,000,000.00
Treasury Value: $600,000,000.00
Bank Value: $300,000,000.00


# Task 1: Portfolio Construction and Expected Return Estimation



In [353]:
import pandas as pd

df = pd.read_excel("../data/stock_prices.xlsx")
print (df.head())



        Date     FFC  SAZEW   OGDC    MEBL    LUCK
0 2023-01-02  100.56  48.82  80.22  100.70  439.25
1 2023-01-03  100.19  48.55  79.94   99.76  431.40
2 2023-01-04  101.38  48.22  78.55   98.86  432.61
3 2023-01-05  101.94  50.81  79.77   97.98  430.74
4 2023-01-06  102.30  50.71  82.31   96.43  433.11


In [354]:
print("Shape:", df.shape)
print("\nColumns:", df.columns)

print("\nData Types:")
print(df.dtypes)

print("\nMissing Values:")
print(df.isnull().sum())

Shape: (807, 6)

Columns: Index(['Date', 'FFC', 'SAZEW', 'OGDC', 'MEBL', 'LUCK'], dtype='object')

Data Types:
Date     datetime64[ns]
FFC             float64
SAZEW           float64
OGDC            float64
MEBL            float64
LUCK            float64
dtype: object

Missing Values:
Date     0
FFC      0
SAZEW    0
OGDC     0
MEBL     0
LUCK     5
dtype: int64


In [355]:
df['Date'] = pd.to_datetime(df['Date'], errors='coerce')
df = df.sort_values(by='Date')

# Forward fill (use last available price)
df = df.fillna(method='ffill')

# Backward fill (for initial missing values)
df = df.fillna(method='bfill')

# Converting Date column into index so time-series calculations (like returns) work properly
df = df.set_index("Date")

C:\Users\syedy\AppData\Local\Temp\ipykernel_28644\3059391915.py:5: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df = df.fillna(method='ffill')
C:\Users\syedy\AppData\Local\Temp\ipykernel_28644\3059391915.py:8: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df = df.fillna(method='bfill')


In [356]:
print("Missing after cleaning:")
print(df.isnull().sum())

Missing after cleaning:
FFC      0
SAZEW    0
OGDC     0
MEBL     0
LUCK     0
dtype: int64


In [357]:
print(df.info())
print(df.describe())

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 807 entries, 2023-01-02 to 2026-04-03
Data columns (total 5 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   FFC     807 non-null    float64
 1   SAZEW   807 non-null    float64
 2   OGDC    807 non-null    float64
 3   MEBL    807 non-null    float64
 4   LUCK    807 non-null    float64
dtypes: float64(5)
memory usage: 37.8 KB
None
              FFC        SAZEW        OGDC        MEBL        LUCK
count  807.000000   807.000000  807.000000  807.000000   807.00000
mean   267.387993   848.674226  169.610409  243.729244   687.38368
std    170.969148   661.413807   71.844633  121.191453   314.15198
min     90.970000    44.710000   73.690000   82.960000   287.68000
25%    103.790000   154.060000  101.955000  135.490000   435.67000
50%    180.540000   988.840000  138.340000  235.040000   583.29000
75%    399.810000  1263.835000  224.605000  309.950000   866.51500
max    673.160000  2455.810000  333.

In [358]:
# Importing necessary libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import minimize

### Step 1: Calculating Daily Returns of Stocks

In [359]:
# Calculate daily returns for each stock
returns = df.pct_change().dropna()  # Percentage change, drop first NaN row
print("Daily Returns (first 5 rows):")
print(returns.head())
print("\nShape:", returns.shape)

Daily Returns (first 5 rows):
                 FFC     SAZEW      OGDC      MEBL      LUCK
Date                                                        
2023-01-03 -0.003679 -0.005531 -0.003490 -0.009335 -0.017871
2023-01-04  0.011877 -0.006797 -0.017388 -0.009022  0.002805
2023-01-05  0.005524  0.053712  0.015532 -0.008901 -0.004323
2023-01-06  0.003531 -0.001968  0.031842 -0.015820  0.005502
2023-01-09 -0.008113 -0.042201  0.030616 -0.014311 -0.021981

Shape: (806, 5)


### Step 2: Computin the Expected Returns

In [360]:
# Calculate average daily returns and annualize them (252 trading days)
# This gives expected annual return for each stock
expected_returns = returns.mean() * 252 
print("Expected Annual Returns:")
for stock, ret in expected_returns.items():
    print(f"{stock}: {ret:.4f} ({ret*100:.2f}%)")

Expected Annual Returns:
FFC: 0.5409 (54.09%)
SAZEW: 1.2189 (121.89%)
OGDC: 0.4567 (45.67%)
MEBL: 0.5377 (53.77%)
LUCK: 0.2513 (25.13%)


#### Expected Returns Interpretation

The expected annual returns for each stock have been calculated using historical daily data.

From the results, it can be seen that some stocks like SAZEW and FFC are giving relatively higher returns, while others like LUCK are more moderate.

This variation is useful because it allows us to balance the portfolio by combining high-return (but possibly riskier) stocks with more stable ones.

Overall, these expected returns will be used in the next steps to construct and evaluate the portfolio.


### Step 3: Computing the Covariance Matrix

In [361]:
cov_matrix = returns.cov() * 252  # Annualize
print("Annualized Covariance Matrix:")
print(cov_matrix)

Annualized Covariance Matrix:
            FFC     SAZEW      OGDC      MEBL      LUCK
FFC    0.096357  0.033359  0.044138  0.031169  0.039514
SAZEW  0.033359  0.252884  0.052687  0.046683  0.064913
OGDC   0.044138  0.052687  0.154117  0.043689  0.060952
MEBL   0.031169  0.046683  0.043689  0.101925  0.040539
LUCK   0.039514  0.064913  0.060952  0.040539  0.312882


#### Interpretating the Matrix

In this step, weights have been assigned to each asset while keeping all the given constraints in mind.

The total equity portion has been fixed at 55%, with the remaining allocated to Treasury Bills (30%) and Bank Deposits (15%).

Within equities, weights have been distributed across the selected stocks in a way that respects both individual stock limits and sector exposure limits.

More stable sectors like banking have been given relatively higher weights, while riskier stocks have been kept at lower allocations.

Overall, this allocation ensures that the portfolio follows the investment policy while still aiming for good returns.

### Step 4: Assigning Portfolio Weights while keeping constraints in mind


In this step, the portfolio weights are assigned in two phases.

**Phase 1: Initial Allocation (Manual)**
First, a valid set of weights is assigned manually while strictly following the constraints given in Annexure A. The goal here is to create a feasible portfolio before moving to optimization.

While assigning weights, the following constraints were considered:
- Total equity allocation must be **55%** of the portfolio
- Treasury Bills = **30%**, Bank Deposits = **15%**
- Each stock cannot exceed **25% of the total portfolio**
- Sector exposure limits:
  - Banking ≤ 40% of equity
  - Fertilizer ≤ 20% of equity
  - Oil & Gas ≤ 25% of equity
  - Other ≤ 25% of equity

  Since total portfolio = PKR 2 billion:

- Equity portion (55%) = PKR 1.1 billion  
- Maximum exposure per stock = 25% of total = PKR 500 million  
- Maximum "Other" sector exposure = 25% of equity = PKR 275 million  

Special attention was given to the **“Other” sector**, since two of the selected stocks (SAZEW and LUCK) fall under it. Their combined weight will be kept within the allowed limit.

We are doing this as this phase ensures that the portfolio is both valid and compliant before moving forward.

**Phase 2: Optimization (Task 2)**
After constructing a valid portfolio, the next step will be to optimize the weights to achieve the **maximum Sharpe ratio**, while still respecting all the constraints.



In [362]:
# Weights as fractions (sum to 1.0)
weights = {
     "FFC": 0.11,    # Fertilizer (at limit 20% of equity)
    "OGDC": 0.13,   # Oil & Gas (strong allocation)
    "MEBL": 0.20,   # Banking (stable, higher weight)
    "SAZEW": 0.04,  # Other (high return but risky)
    "LUCK": 0.07,    # Other (controlled exposure)
    'T_Bills': 0.30,  # 30%
    'Deposits': 0.15  # 15% (10% commercial + 5% Islamic)
}
print("Portfolio Weights:")
for asset, w in weights.items():
    print(f"{asset}: {w:.4f} ({w*100:.2f}%)")
print(f"Total: {sum(weights.values()):.4f}")  # Should be 1.0

Portfolio Weights:
FFC: 0.1100 (11.00%)
OGDC: 0.1300 (13.00%)
MEBL: 0.2000 (20.00%)
SAZEW: 0.0400 (4.00%)
LUCK: 0.0700 (7.00%)
T_Bills: 0.3000 (30.00%)
Deposits: 0.1500 (15.00%)
Total: 1.0000


### Step 5: Calculating Portfolio Return

In [363]:
# Expected returns including fixed income
expected_all = {
    "FFC": expected_returns["FFC"],
    "OGDC": expected_returns["OGDC"],
    "MEBL": expected_returns["MEBL"],
    "SAZEW": expected_returns["SAZEW"],
    "LUCK": expected_returns["LUCK"],
    "T_Bills": 0.15,
    "Deposits": 0.12
}

portfolio_return = 0

for asset in weights:
    portfolio_return += weights[asset] * expected_all[asset]

print("Portfolio Expected Return:", portfolio_return)

Portfolio Expected Return: 0.3557544600896844


#### Interpretation of Portfolio Return


The expected return of the portfolio is approximately **35.58%**.

This shows that the portfolio is generating a strong return, mainly due to the equity portion, while fixed-income assets like Treasury Bills and Deposits provide stable contributions.

Since a significant portion of the portfolio is invested in equities, higher returns are expected, but the presence of safer assets helps balance the overall structure.

Overall, the portfolio seems to achieve a good mix of growth and stability.

## Task 1 Deliverables

The following outputs were obtained after constructing the initial portfolio:

- **Selected Stocks:** FFC, SAZEW, OGDC, MEBL, LUCK  
- **Dataset:** Daily closing prices (past 3 years) collected and structured using Pandas  
- **Portfolio Allocation (Initial Weights):**

| Asset            | Weight (%) |
|------------------|-----------|
| FFC              | 11.00%    |
| SAZEW            | 4.00%     |
| OGDC             | 13.00%    |
| MEBL             | 20.00%    |
| LUCK             | 7.00%     |
| Treasury Bills   | 30.00%    |
| Bank Deposits    | 15.00%    |

- **Expected Portfolio Return:** 35.58%

This portfolio satisfies all the constraints defined in Annexure A and serves as the base case before optimization.

# Task 2: Portfolio Risk, Sharpe Ratio optimization

### Step 1: Calculating Portfolio Risk (Standard Deviation)

In [364]:
# Only stock weights (same order as covariance matrix)
stock_weights = np.array([0.11, 0.04, 0.13, 0.20, 0.07])

portfolio_variance = np.dot(stock_weights.T, np.dot(cov_matrix, stock_weights))
portfolio_std = np.sqrt(portfolio_variance)

print("Portfolio Risk (Std Dev):", portfolio_std)

Portfolio Risk (Std Dev): 0.13962700982767454


#### Interpretation of Portfolio Risk

The portfolio standard deviation came out to be around **13.96%**, which basically shows how much the returns can fluctuate.

Since more than half of the portfolio is in equities (55%), some level of volatility is expected. However, the presence of Treasury Bills and bank deposits helps reduce overall risk.

So overall, the portfolio is not extremely risky, but it’s not super safe either — it’s somewhere in the middle, which makes sense given the objective of earning returns while still controlling risk.

### Step 2: Computing the Sharpe Ratio

In [365]:
risk_free_rate = 0.15  # T-Bills rate has been used as risk-free rate according to guidelines given

sharpe_ratio = (portfolio_return - risk_free_rate) / portfolio_std

print("Sharpe Ratio:", sharpe_ratio)

Sharpe Ratio: 1.4736007047893052


#### Interpretation of Sharpe Ratio

The Sharpe ratio is around **1.47**, which is a pretty good result.

It shows that the portfolio is generating a strong return compared to the level of risk being taken. In simple terms, we are getting more return per unit of risk.

Since the risk-free rate (T-Bills) is 15%, and our portfolio is giving much higher returns, this indicates that the portfolio is performing well and is efficiently using the available funds.

Overall, this suggests that the portfolio is well balanced and achieving a good trade-off between risk and return.

### Step 3: Optimizing the Portfolio

In [366]:

num_portfolios = 5000 # Number of random portfolios to simulate

best_sharpe = -100 # Starting with a very low Sharpe ratio to ensure any valid portfolio will be better
best_weights = None # To store the best weights corresponding to the highest Sharpe ratio
best_return = 0 # To store the expected return of the best portfolio
best_risk = 0 # To store the risk of the best portfolio

for i in range(num_portfolios):

    # Generate random weights for 5 stocks and scale to 55% equity portion
    w = np.random.random(5)
    w = w / sum(w)      # make weights sum to 1
    w = w * 0.55        # scale to 55% equity portion

    # Assign stock weights clearly
    ffc = w[0]    # Fertilizer
    sazew = w[1]  # Other
    ogdc = w[2]   # Oil & Gas
    mebl = w[3]   # Banking
    luck = w[4]   # Other

    # -----------------------------
    # Constraint Checks - basically if a stock is greater than its sector limit, skip this portfolio (simple as that)
    # -----------------------------

    # Fertilizer <= 20% of equity = 0.20 * 0.55 = 0.11
    if ffc > 0.11:
        continue

    # Oil & Gas <= 25% of equity = 0.25 * 0.55 = 0.1375
    if ogdc > 0.1375:
        continue

    # Banking <= 40% of equity = 0.40 * 0.55 = 0.22
    if mebl > 0.22:
        continue

    # Other sector <= 25% of equity = 0.1375
    if (sazew + luck) > 0.1375:
        continue

    # -----------------------------
    # Portfolio Return
    # -----------------------------
    port_return = (
        ffc   * expected_returns["FFC"] +
        sazew * expected_returns["SAZEW"] +
        ogdc  * expected_returns["OGDC"] +
        mebl  * expected_returns["MEBL"] +
        luck  * expected_returns["LUCK"] +
        0.30  * 0.15 +   # T-Bills
        0.15  * 0.12     # Deposits
    )

    # -----------------------------
    # Portfolio Risk
    # -----------------------------
    stock_weights = np.array([ffc, sazew, ogdc, mebl, luck])

    port_var = np.dot(stock_weights.T, np.dot(cov_matrix, stock_weights))
    port_std = np.sqrt(port_var)

    # -----------------------------
    # Sharpe Ratio
    # -----------------------------
    sharpe = (port_return - 0.15) / port_std

    # -----------------------------
    # Save Best Portfolio
    # -----------------------------
    if sharpe > best_sharpe:
        best_sharpe = sharpe
        best_weights = stock_weights
        best_return = port_return
        best_risk = port_std

# -----------------------------
# Print Final Best Portfolio
# -----------------------------
print("Optimal Portfolio Found")
print("-----------------------")
print(f"Best Sharpe Ratio: {best_sharpe:.4f}")
print(f"Expected Return: {best_return:.4f} ({best_return*100:.2f}%)")
print(f"Portfolio Risk: {best_risk:.4f} ({best_risk*100:.2f}%)")

print("\nOptimal Stock Weights:")
stocks = ["FFC", "SAZEW", "OGDC", "MEBL", "LUCK"]
for i in range(5):
    print(f"{stocks[i]}: {best_weights[i]:.4f} ({best_weights[i]*100:.2f}%)")

print("T-Bills: 0.3000 (30.00%)")
print("Deposits: 0.1500 (15.00%)")

Optimal Portfolio Found
-----------------------
Best Sharpe Ratio: 1.7001
Expected Return: 0.3878 (38.78%)
Portfolio Risk: 0.1399 (13.99%)

Optimal Stock Weights:
FFC: 0.1034 (10.34%)
SAZEW: 0.0768 (7.68%)
OGDC: 0.1354 (13.54%)
MEBL: 0.1903 (19.03%)
LUCK: 0.0441 (4.41%)
T-Bills: 0.3000 (30.00%)
Deposits: 0.1500 (15.00%)


#### Interpretation of the Optimal Portfolio calculated

After running multiple portfolio combinations, the portfolio with the highest Sharpe ratio was selected.

The optimal portfolio gives a Sharpe ratio of **1.70**, which is higher than our initial portfolio. This means the optimized portfolio is providing better return for the same level of risk.

The expected return increased to around **38.78%**, while the risk remained almost the same at around **14%**. This shows that the portfolio became more efficient without taking additional risk.

Looking at the weights, more allocation has been shifted towards stocks like MEBL and OGDC, which offer a better balance of return and stability, while exposure to riskier stocks like LUCK has been reduced.

Overall, the optimized portfolio improves performance while still staying within all the given constraints, making it a better choice compared to the initial allocation.

## Task 2 Deliverables

After evaluating portfolio risk and performing optimization, the following results were obtained:

### Optimal Portfolio (Maximum Sharpe Ratio)

| Asset            | Weight (%) |
|------------------|-----------|
| FFC              | 10.34%    |
| SAZEW            | 7.68%     |
| OGDC             | 13.54%    |
| MEBL             | 19.03%    |
| LUCK             | 4.41%     |
| Treasury Bills   | 30.00%    |
| Bank Deposits    | 15.00%    |

### Portfolio Performance

- **Expected Return:** 38.78%  
- **Portfolio Risk (Std Dev):** 13.99%  
- **Sharpe Ratio:** 1.70  

The optimized portfolio improves the risk-adjusted return while remaining within all sector and allocation constraints.